# **Reinforcement Learning Agent For CNN Hyper Parameter Tunning**
- here we will be considered :
    - leraning rate
    - batch size
    - 

In [1]:
import torch # main deep learning library
import torch.nn as nn # contains neural network layers and loss functions
import torch.optim as optim # optimization algorithms (SGD, Adam, etc.)

import torchvision # datasets + transforms for images
import torchvision.transforms as transforms
from torch.utils.data import random_split

import matplotlib

print(torch.__version__)
print(torchvision.__version__)
print(matplotlib.__version__)

# If GPU is available → use it
# Otherwise → fallback to CPU
     
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device : ", device)

2.5.1
0.20.1
3.10.9
Using device :  cpu


### Data Preproccesing

In [2]:
# Transforms
transform = transforms.Compose([
    # Converts PIL Image or NumPy array (pixel values 0–255) to a PyTorch tensor, Also scales pixel values from 0–255 → 0–1 (float)
    transforms.ToTensor(), 
    
    # Standardizes the tensor values to make training more stable and help the network converge faster, normalized_pixel=(pixel - mean)/std
    # This scales the pixel range from 0–1 → -1–1, which is often better for neural networks with activation functions like tanh or ReLU.
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

### Dataloaders

In [3]:
from torch.utils.data import DataLoader, Subset

# Download train data set
trainset = torchvision.datasets.MNIST(
    root="./data", # folder to store dataset
    train=True,  # training data
    download=True, # download if not exists
    transform=transform # apply preprocessing
)

# Download test data set
testset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create validations set from training set
train_size = int(0.8 * len(trainset))
val_size = len(trainset) - train_size

train_data, val_data = random_split(trainset, [train_size, val_size])

# trainloader = DataLoader(
#     train_data,
#     batch_size = 64,
#     shuffle=True
# )

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

valloader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False
)

# Fast train loader
subset_indices = list(range(10000))  # use only first 10k samples
train_subset = Subset(train_data, subset_indices)

trainloader_fast = DataLoader(train_subset, batch_size=64, shuffle=True)

def get_train_loader(batch_size):
    return DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True
    )

### Modified Le-Net CNN

In [28]:
class ModifiedLeNet(nn.Module):

    # constructor → define layers >>> batch(64 images at once) → conv → activation → pooling → fc → output
    def __init__(self, dropout_rate):
        super().__init__()

        # ================= CONVOLUTIONAL BLOCK (FEATURE EXTRACTOR) ==================================

        # This part extracts image features
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
    
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # ================= FULLY CONNECTED BLOCK (CLASSIFIER (Head))==================================

        # This part performs classification
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16*4*4, 120),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(120, 84),
            nn.ReLU(),

            nn.Linear(84, 10)
        )

    # forward pass defines data flow
    # input image > conv layers → feature maps > flatten > fully connected layers  > class scores
    '''
    This is the actual computation pipeline of the model, When we call > output = model(images), PyTorch internally does:model.forward(images)
    So this function is the execution path
    
    def forward(self, x): >>> x = input batch(shape : [batch, channels, height, width])
    x = self.conv_layers(x) >>> This sends the input through : Conv → ReLU → Pool → Conv → ReLU → Pool → Conv → ReLU, 
    So after this line > x = extracted feature maps
    x = self.fc_layers(x) >>> Now x goes into > Flatten → Linear → ReLU → Linear
    return x >>> Returns predictions > Example output for one image: [0.2, 1.1, 0.3, 5.4, 0.1, 0.0, 0.2, 0.1, 0.0, 0.2] 
    > Largest value index = predicted digit
    '''
    def forward(self, x):
        
        # pass input through conv layers
        x = self.conv_layers(x)
        
        # pass result through fully connected layers
        x = self.fc_layers(x)
        
        return x

### CREATE A FUNCTION TO TRAIN MODEL FOR Q-LEARNING ALGORITHM

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim

# epochs - number of rounds that model train on entire dataset
# lr - lr of cnn optimizer

def train_model(train_loader, device, lr, batch_size, dropout, epochs, stop_acc=97):

    train_loader = get_train_loader(batch_size)

    model = ModifiedLeNet(dropout).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    
    loss_history = []
    acc_history = []
    
    # Train model for 1 epoch only for testing
    for epoch in range(epochs):

        # ---------------- TRAIN ----------------
        model.train()
        
        running_loss = 0
        correct = 0
        total = 0

        # iterate over batches, When load MNIST using PyTorch dataset loaders, each batch gives: (images, labels) 64 sets(batches)
        # enumerate add index for each batch here we may have 938 batch all will be indexed as 1, 2, 3, ...
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() # summ all loss for each batch (938 losses), bcs we need entire dataset loss

            epoch_loss = running_loss / len(train_loader)
            loss_history.append(epoch_loss)
            
            #if (i+1) % 100 == 0:  # print every 100 batches
            #    print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}")

        # ---------------- EVALUATE ----------------
        model.eval()
        correct = 0
        total = 0
        
        # no gradients needed during testing
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                
                output = model(images)
                # le-net class return x by forward method, but it is 10 porbabilitys list, let get highetst prob's index it is the final prediction
                _, predicted = torch.max(output, 1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
            epoch_acc = 100 * correct / total
            acc_history.append(epoch_acc)

            print(f"Epoch {epoch+1} Loss: {running_loss/len(train_loader):.4f}, Accuracy: {epoch_acc}")

            # ---------------- EARLY STOPPING ----------------        
            if epoch_acc >= stop_acc:
                print("*** Early stopping triggered — model already good!")
                break
                
        # print(f"\nFinal Test Accuracy: {100 * correct / total:.2f}%")
        
    return model, loss_history, acc_history

### State encoding

In [30]:
import itertools

# 4 x 3 x 3 = 36 - total 
# Q_table size = [36 states × actions]

state_space = list(itertools.product(
    learning_rates,
    batch_sizes,
    dropouts
))

num_states = len(state_space)
print("Total states : ", num_states)

Total states :  36


### Define Q-Learning Table

In [31]:
import numpy as np
import random

learning_rates = [0.0001, 0.001, 0.01, 0.1]
batch_sizes = [32, 64, 128]
dropouts = [0.0, 0.25, 0.5]

# -1: decrease, 0: keep, 1: increase
actions = [0,1,2,3,4,5]
num_actions = len(actions)

# Define Q table
Q_table = np.zeros((num_states, num_actions))

### Hyperparameters for Q-learning

In [32]:
alpha = 0.1          # learning rate > how fast Q values update

gamma = 0.9          # discount factor > how much the agent cares about future rewards

epsilon = 1.0        # exploration rate > probability of taking random action

epsilon_decay = 0.95 # after each episode, exploration reduces

epsilon_min = 0.05   # minimum exploration -> agent still explores 5% forever

max_steps = 2        # maximum steps per episode (how long agent can move before episode ends)

### Helper Functions for Q-Learning

In [38]:
# State <-> Index mapping (return the index of passed lr value in the above defined list)
def get_state_index(state):
    return state_space.index(state)

# Decode state
def decode_state(index):
    return state_space[index]
    
def take_action(state, action):

    lr, batch, dropout = state

    lr_idx = learning_rates.index(lr)
    batch_idx = batch_sizes.index(batch)
    drop_idx = dropouts.index(dropout)
    
    if action == 0 and lr_idx < len(learning_rates)-1 :
        lr_idx += 1
    elif action == 1 and lr_idx > 0 :
        lr_idx -= 1
        
    elif action == 2 and batch_idx < len(batch_sizes)-1 : #
       batch_idx += 1
    elif action == 3 and batch_idx > 0 : #
       batch_idx -= 1

    elif action == 4 and drop_idx < len(dropouts)-1:
        drop_idx += 1
    elif action == 5 and drop_idx > 0:
        drop_idx -= 1

    new_state = (
        learning_rates[lr_idx],
        batch_sizes[batch_idx],
        dropouts[drop_idx]
    )
    
    return new_state

### Q-Learning Training Loop

In [39]:
# Use proper max_steps=1 – 3, epcohs=3 – 8, episodes=20 – 50 values

'''
---EPISODE---
One complete run of an agent from start -> until it stops
Episode = one full experience / one game / one attempt
Start -> agent acts -> gets rewards -> reaches goal or fails > END
That whole journey = 1 episode

if cnn epoches = 10, RL episodes = 20, steps = 25 >>> in each RL episode CNN train 25 times, like wise 20 times RL do same thing, in each CNN training
time it will train 10 times on entire dataset

---STEPS---
how long agent can move before episode ends
in each step CNN will be trained , retuern acc, update q table
'''

# ========== NOTE ==================
'''
until we find optimusm hps we use small training fat set , after found we trin again model with entire dat set and proper other valus, but now
epochs = 2-3 
steps = 2–4 
episodes = 20–50

entir dataset + epoch(10–20) + found lr
'''
# ========= Parameters ==============

episodes = 3  # how many trials the agent runs (20)
epochs= 2
max_steps= 3  # already define but for clarity

# Set table values to zero since i run several times this cell for testing
Q_table = np.zeros((num_states, num_actions))

# Episodes × Steps = 5×10=50 times, The CNN training function is called 50 times, if cnn epochs=5, in each training data set used 5 times(5x10x5)

for episode in range(episodes):
    
    # Start with random lr
    state = random.choice(state_space)
    state_index = get_state_index(state)
    
    lr, batch_size, dropout = state

    print(f"Episode {episode+1} started with random learning_rate = {lr} & state_index = {state_index} state = {state}")

    # Initialize a new CNN model for each episode
    # model = ModifiedLeNet().to(device)
    
    # Train model and get reward(acc)
    _, losses, accuracies = train_model(
        train_loader= trainloader_fast,
        device=device, 
        lr=lr,
        batch_size=batch_size,
        dropout = dropout,
        epochs=epochs
    )
    
    step = 1

    # Do untill the reach max steps 1, 2, 3, ..., max_steps
    while step <= max_steps:
        print(f"\nEPISODE {episode+1} STEP {step} started")

        # store previously trained model accuracy
        old_accuracy = accuracies[-1]
        
        # Choose action: epsilon-greedy
        if random.uniform(0, 1) < epsilon:
            # action index means the column number of the choosed action
            action_index = random.randint(0, len(actions)-1) # 0, 1, 2
            print(f"Explore action_index : {action_index} (0, 1, ,2)")
        else:
            # np.argmax([...]) returns the index of the maximum value in that Q_table[state_index] row
            # if all values are equal (all zeros), np.argmax returns index of the first occurrence of the maximum value(0)
            # action index means the column number of the choosed action
            action_index = np.argmax(Q_table[state_index]) # exploit, range 0-2, state index means row of the currently used lr 
            print(f"Exploit action_index : {action_index} (0, 1, ,2)")
            
        action = actions[action_index] # -1, 0, 1
        print(f"ACTION : {action} (-1, 0, 1)")
        
        
        new_state = take_action(state, action)
        new_state_index = get_state_index(new_state)
        
        new_lr, new_batch, new_dropout = new_state
        
        #print(f"new_state_index ={new_state_index} (old - {state_index})  , new_lr = {new_lr} (old - {lr})")

        # Train CNN with new learning rate
        _, losses, accuracies = train_model(
            trainloader_fast, 
            device, 
            new_lr,
            new_batch,
            new_dropout,
            epochs
        )
        new_accuracy = accuracies[-1]
        
        reward = new_accuracy - old_accuracy
        # reward = new_accuracy / 100
        print(f"new_accuracy ={new_accuracy} , reward = {reward}")

        # Update Q-table
        Q_value = Q_table[state_index, action_index] + alpha * (reward + gamma * np.max(Q_table[new_state_index])- Q_table[state_index, action_index])
        Q_table[state_index, action_index] = Q_value
        print(f"Q_value = {Q_value} added to ({state_index}, {action_index})")
        
        # move state forward
        state = new_state
        state_index = new_state_index
        accuracy = new_accuracy

        step += 1

        # print("Q-table:\n", Q_table)
        print(f"Step {episode+1}.{step}/{max_steps} done\n")
    
    # Prevent agent keeps exploring / exploit forever, decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    print("NEW EPSILON : ", epsilon)
        
    print(f"Episode {episode+1} DONE ( {max_steps} steps used), Final lr : {lr} & accuracy : {accuracy}, Reward: {reward:.2f}%")
    print("Q-table:\n", Q_table, "\n===============================\n")

print("================== DONE =====================")

Episode 1 started with random learning_rate = 0.1 & state_index = 28 state = (0.1, 32, 0.25)
Epoch 1 Loss: 0.8718, Accuracy: 90.93
Epoch 2 Loss: 0.1786, Accuracy: 96.4

EPISODE 1 STEP 1 started
Explore action_index : 2 (0, 1, ,2)
ACTION : 2 (-1, 0, 1)
Epoch 1 Loss: 1.4618, Accuracy: 62.63
Epoch 2 Loss: 0.3114, Accuracy: 90.03
new_accuracy =90.03 , reward = -6.3700000000000045
Q_value = -0.6370000000000005 added to (28, 2)
Step 1.2/3 done


EPISODE 1 STEP 2 started
Explore action_index : 1 (0, 1, ,2)
ACTION : 1 (-1, 0, 1)
Epoch 1 Loss: 2.3044, Accuracy: 10.96
Epoch 2 Loss: 2.2952, Accuracy: 14.58
new_accuracy =14.58 , reward = -75.45
Q_value = -7.545000000000001 added to (31, 1)
Step 1.3/3 done


EPISODE 1 STEP 3 started
Explore action_index : 1 (0, 1, ,2)
ACTION : 1 (-1, 0, 1)
Epoch 1 Loss: 2.3069, Accuracy: 9.94
Epoch 2 Loss: 2.3062, Accuracy: 9.96
new_accuracy =9.96 , reward = -4.619999999999999
Q_value = -0.46199999999999997 added to (22, 1)
Step 1.4/3 done

NEW EPSILON :  0.95
Epis

### Inspect Q-table

In [40]:
print("Q-table after training:")
for i, j in enumerate(state_space):
    print(f"state {j}: {Q_table[i]}")

Q-table after training:
state (0.0001, 32, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.0001, 32, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.0001, 32, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.0001, 64, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.0001, 64, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.0001, 64, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.0001, 128, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.0001, 128, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.0001, 128, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.001, 32, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.001, 32, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.001, 32, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.001, 64, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.001, 64, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.001, 64, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.001, 128, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.001, 128, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.001, 128, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.01, 32, 0.0): [0. 0. 0. 0. 0. 0.]
state (0.01, 32, 0.25): [0. 0. 0. 0. 0. 0.]
state (0.01, 32, 0.5): [0. 0. 0. 0. 0. 0.]
state (0.01, 64, 0.0): [0. 0. 0. 

### Select best learning rate

In [42]:
best_state_index = np.argmax(Q_table.max(axis=1))
best_state = state_space[best_state_index]

best_lr, best_batch, best_dropout = best_state

print(f"Best state found by Q-learning: {best_state}")
print(f"Best learning rate: {best_lr}")
print(f"Best batch size: {best_batch}")
print(f"Best dropout: {best_dropout}")

Best state found by Q-learning: (0.01, 64, 0.5)
Best learning rate: 0.01
Best batch size: 64
Best dropout: 0.5


### Test model with new lr given by RL Agent

In [ ]:
learning_rate = best_lr
epochs = 

model = ModifiedLeNet().to(device)
_, losses, accuracies = train_model(train_loader=train_loader, device=device,model=model, lr=i, epochs=epochs)
print(f"Acc : {accuracies[-1]}")